In [1]:
import pandas as pd
import json
from drug_price_optimization.agents.data_foundation_agent import DataFoundationAgent

c:\Users\USER\anaconda3\envs\agno\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [2]:
# Example usage in notebook:

# Load your NADAC dataset
nadac_df = pd.read_csv('../data/raw/cleaned_nadac-national-average-drug-acquisition-cost-05-14-2025.csv')

In [9]:
nadac_df.head(2)

,NDC Description,NDC,NADAC Per Unit,Effective Date,Pricing Unit,Pharmacy Type Indicator,OTC,Explanation Code,Classification for Rate Setting,Corresponding Generic Drug NADAC Per Unit,Corresponding Generic Drug Effective Date,As of Date
0,ABILIFY 10 MG TABLET,59148000813,18.68756,12/18/2024,EA,C/I,N,1,B,0.11884,12/18/2024,01/01/2025
1,ABILIFY 15 MG TABLET,59148000913,18.71385,12/18/2024,EA,C/I,N,1,B,0.12704,12/18/2024,01/01/2025


In [3]:
try:
    agent = DataFoundationAgent()
    print("Agent instantiated:", agent)
except Exception:
    print("Agent instantiation failed. Traceback:")

Agent instantiated: DataFoundationAgent(model='microsoft/DialoGPT-small', name='DataFoundationAgent', agent_id=None, introduction=None, user_id=None, session_id=None, session_name=None, session_state=None, search_previous_sessions_history=False, num_history_sessions=None, context=None, add_context=False, resolve_context=True, memory=None, enable_agentic_memory=False, enable_user_memories=False, add_memory_references=None, enable_session_summaries=False, add_session_summary_references=None, add_history_to_messages=False, num_history_responses=None, num_history_runs=3, knowledge=None, knowledge_filters=None, enable_agentic_knowledge_filters=None, add_references=False, retriever=None, references_format='json', storage=None, extra_data=None, tools=[<DataFoundationToolkit name=data_foundation_toolkit functions=['clean_drug_names', 'normalize_pricing', 'standardize_categories', 'compute_time_features', 'compute_summary_stats', 'create_drug_profiles']>], show_tool_calls=True, tool_call_limit=

In [4]:
result = agent.process_nadac_data(nadac_df)
print("Processing succeeded. Summary stats keys:", result.get("summary_statistics", {}).keys())

🧹 Starting data cleaning and normalization...
✅ Cleaned 22747 records
📊 Computed features for 1184 unique drugs
🗃️ Created 1184 structured drug profiles
✨ Data foundation processing complete!
Processing succeeded. Summary stats keys: dict_keys(['total_records', 'unique_drugs', 'date_range', 'brand_generic_split', 'pharmacy_type_split', 'price_stats', 'data_quality'])


In [7]:
# Save structured output for next agent
with open('../outputs/foundation_output.json', 'w') as f:
    json.dump(result, f, indent=2)

print(f"Processed {result['processing_status']['total_output_profiles']} drug profiles")

Processed 1184 drug profiles


In [13]:
result['drug_profiles'][0]  # Display first 2 drug profiles

{'drug_id': 'ndc_2322730',
 'ndc': 2322730,
 'drug_name': 'STRATTERA 10 MG CAPSULE',
 'classification': {'brand_generic': 'Brand', 'otc_rx': 'Prescription'},
 'pricing_data': {'current_avg_cost': 12.692232999999998,
  '30_day_moving_avg': 12.70385,
  'price_volatility': 0.00019419683865674718,
  'min_price': 12.66775,
  'max_price': 12.70714,
  'price_trend': 'stable'},
 'data_quality': {'record_count': 20,
  'date_range': {'start': '2024-12-18', 'end': '2025-04-23'},
  'completeness': 1.0,
  'last_updated': '2025-04-23'}}

In [17]:
from drug_price_optimization.agents.market_intelligence_agent import MarketIntelligenceAgent

In [18]:
try:
    agent1 = MarketIntelligenceAgent()
    print("Agent instantiated:", agent1)
except Exception:
    print("Agent instantiation failed. Traceback:")

Agent instantiated: MarketIntelligenceAgent(model='microsoft/DialoGPT-medium', name='MarketIntelligenceAgent', agent_id=None, introduction=None, user_id=None, session_id=None, session_name=None, session_state=None, search_previous_sessions_history=False, num_history_sessions=None, context=None, add_context=False, resolve_context=True, memory=None, enable_agentic_memory=False, enable_user_memories=False, add_memory_references=None, enable_session_summaries=False, add_session_summary_references=None, add_history_to_messages=False, num_history_responses=None, num_history_runs=3, knowledge=None, knowledge_filters=None, enable_agentic_knowledge_filters=None, add_references=False, retriever=None, references_format='json', storage=None, extra_data=None, tools=[<MarketBenchmarkingTool name=market_benchmarking_tool functions=['calculate_market_benchmarks', 'calculate_drug_benchmark_ranges']>, <CompetitiveAnalysisTool name=competitive_analysis_tool functions=['analyze_competition_level']>, <Mark

In [19]:
with open('../outputs/foundation_output.json', 'r') as f:
    foundation_output = json.load(f)

In [ ]:
res = agent1.analyze_market_intelligence(foundation_output)

📊 Starting market intelligence analysis...
Debug: Found 1183 valid prices out of 1184 drugs
📈 Calculated market benchmarks for 1184 drugs
🎯 Completed competitive analysis for 1184 drugs
📊 Completed market trend analysis
✨ Market intelligence analysis complete!


In [23]:
# Save structured output for next agent
with open('../outputs/intelligence_output.json', 'w') as f:
    json.dump(res, f, indent=2)

print(f"Analyzed {res['processing_status']['drugs_processed']} drugs")
print(f"Market sentiment: {res['market_overview']['market_trends']['market_sentiment']}")

Analyzed 1184 drugs
Market sentiment: stable


In [25]:
res['drug_intelligence'][0]

{'drug_id': 'ndc_2322730',
 'drug_name': 'STRATTERA 10 MG CAPSULE',
 'ndc': 2322730,
 'classification': {'brand_generic': 'Brand', 'otc_rx': 'Prescription'},
 'market_intelligence': {'benchmark_min_price': 11.423009699999998,
  'benchmark_max_price': 13.961456299999998,
  'market_median': 9.688075999999999,
  'market_mean': 22.89821456048632,
  'price_position_vs_median': 0.31008809179449043,
  'price_quartile': 'above_median',
  'competition_level': 'high',
  'total_competitors': 6,
  'brand_competitors': 6,
  'generic_competitors': 0,
  'avg_competitor_price': 13.698857499999997,
  'price_advantage_pct': 7.348236887638253,
  'competitive_threats': ['Limited competitive pressure detected'],
  'market_opportunities': ['Market-rate pricing - opportunity for differentiation'],
  'market_volatility': 0.00019419683865674718,
  'trend': 'stable'}}

In [3]:
from drug_price_optimization.agents.profitability_agent import ProfitabilityConstraintAgent

try:
    agent2 = ProfitabilityConstraintAgent()
    print("Agent instantiated:", agent2)
except Exception:
    print("Agent instantiation failed. Traceback:")

Agent instantiated: ProfitabilityConstraintAgent(model='microsoft/DialoGPT-medium', name='ProfitabilityConstraintAgent', agent_id=None, introduction=None, user_id=None, session_id=None, session_name=None, session_state=None, search_previous_sessions_history=False, num_history_sessions=None, context=None, add_context=False, resolve_context=True, memory=None, enable_agentic_memory=False, enable_user_memories=False, add_memory_references=None, enable_session_summaries=False, add_session_summary_references=None, add_history_to_messages=False, num_history_responses=None, num_history_runs=3, knowledge=None, knowledge_filters=None, enable_agentic_knowledge_filters=None, add_references=False, retriever=None, references_format='json', storage=None, extra_data=None, tools=[<ProfitabilityCalculationTool name=profitability_calculation_tool functions=['calculate_base_margins', 'calculate_profit_projections']>, <RegulatoryConstraintTool name=regulatory_constraint_tool functions=['apply_regulatory_li

In [4]:
with open('../outputs/intelligence_output.json', 'r') as f:
    intelligence_output = json.load(f)

In [5]:
ans = agent2.calculate_optimal_pricing(intelligence_output)

💰 Starting profitability and constraint analysis...
💡 Generated pricing recommendations for 1184 drugs
✨ Profitability and constraint analysis complete!


In [6]:
# Save structured output for next agent
with open('../outputs/profitability_output.json', 'w') as f:
    json.dump(ans, f, indent=2)

In [7]:
print(f"Generated pricing for {ans['processing_status']['drugs_processed']} drugs")
print(f"Average expected margin: {ans['portfolio_summary']['average_expected_margin']:.1%}")
print(f"Total projected annual profit: ${ans['portfolio_summary']['total_projected_annual_profit']:,.0f}")

Generated pricing for 1184 drugs
Average expected margin: 33.4%
Total projected annual profit: $569,900,343


In [3]:
from drug_price_optimization.agents.strategy_recommendation_agent import StrategyRecommendationAgent

try:
    agent3 = StrategyRecommendationAgent()
    print("Agent instantiated:", agent3)
except Exception:
    print("Agent instantiation failed. Traceback:")

Agent instantiated: StrategyRecommendationAgent(model='microsoft/DialoGPT-medium', name='StrategyRecommendationAgent', agent_id=None, introduction=None, user_id=None, session_id=None, session_name=None, session_state=None, search_previous_sessions_history=False, num_history_sessions=None, context=None, add_context=False, resolve_context=True, memory=None, enable_agentic_memory=False, enable_user_memories=False, add_memory_references=None, enable_session_summaries=False, add_session_summary_references=None, add_history_to_messages=False, num_history_responses=None, num_history_runs=3, knowledge=None, knowledge_filters=None, enable_agentic_knowledge_filters=None, add_references=False, retriever=None, references_format='json', storage=None, extra_data=None, tools=[<PricingStrategyTool name=pricing_strategy_tool functions=['determine_pricing_strategy']>, <RecommendationTool name=recommendation_tool functions=['generate_recommendations']>, <ScenarioAnalysisTool name=scenario_analysis_tool f

In [4]:
with open('../outputs/profitability_output.json', 'r') as f:
    profitability_output = json.load(f)

In [5]:
fin_ans = agent3.generate_strategy_recommendations(profitability_output)

🎯 Starting strategy and recommendation generation...
📋 Generated strategic recommendations for 1184 drugs
✨ Strategy and recommendation generation complete!


In [8]:
with open('../outputs/recommendations_output.json', 'w') as f:
    json.dump(fin_ans, f, indent=2)

In [3]:
from drug_price_optimization.agents.agno_pricing_strategy_agent import StrategyRecommendationAgent

try:
    agent4 = StrategyRecommendationAgent()
    print("Agent instantiated:", agent4)
except Exception:
    print("Agent instantiation failed. Traceback:")

Agent instantiated: StrategyRecommendationAgent(model='microsoft/DialoGPT-medium', name='StrategyRecommendationAgent', agent_id=None, introduction=None, user_id=None, session_id=None, session_name=None, session_state=None, search_previous_sessions_history=False, num_history_sessions=None, context=None, add_context=False, resolve_context=True, memory=None, enable_agentic_memory=False, enable_user_memories=False, add_memory_references=None, enable_session_summaries=False, add_session_summary_references=None, add_history_to_messages=False, num_history_responses=None, num_history_runs=3, knowledge=None, knowledge_filters=None, enable_agentic_knowledge_filters=None, add_references=False, retriever=None, references_format='json', storage=None, extra_data=None, tools=[<PricingStrategyToolkit name=pricing_strategy_toolkit functions=['determine_pricing_strategy', 'calculate_recommended_price', 'generate_rationale', 'assess_confidence', 'determine_implementation_timeline', 'generate_monitoring_p

In [4]:
with open('../outputs/profitability_output.json', 'r') as f:
    profitability_output = json.load(f)

In [5]:
with open('../outputs/intelligence_output.json', 'r') as f:
    intelligence_output = json.load(f)

In [6]:
fin_ans_2 = agent4.process_portfolio(intelligence_output, profitability_output)

🎯 Starting pricing strategy analysis...
💡 Generated strategies for 1184 drugs
📊 Portfolio analysis complete with 1184 strategic recommendations
✨ Pricing strategy processing complete!


In [7]:
fin_ans_2

{'agent': 'StrategyRecommendationAgent',
 'timestamp': '2025-09-17T15:01:26.284570',
 'portfolio_strategy_summary': {'total_drugs_strategized': 1184,
  'strategy_distribution': {'aggressive': 347,
   'balanced': 101,
   'conservative': 736},
  'portfolio_profit_impact': np.float64(nan),
  'high_priority_implementations': 231,
  'average_confidence': np.float64(0.816),
  'key_portfolio_insights': ['Dominant strategy is conservative pricing (736 drugs)',
   'Overall market sentiment is stable',
   'Total projected annual profit: $569,900,342.87']},
 'drug_strategies': [{'drug_id': 'ndc_2322730',
   'drug_name': 'STRATTERA 10 MG CAPSULE',
   'ndc': 2322730,
   'classification': {'brand_generic': 'Brand', 'otc_rx': 'Prescription'},
   'pricing_strategy': {'strategy_type': 'aggressive',
    'recommended_price': 10.9999,
    'current_optimal_price': 11.605830644199997,
    'price_adjustment': -0.6059,
    'price_adjustment_pct': -5.22,
    'strategy_rationale': 'Aggressive pricing strategy s

In [8]:
with open('../outputs/recommendations_output.json', 'w') as f:
    json.dump(fin_ans_2, f, indent=2)